[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [NumPy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)

# Vectorized Math


## What you will be able to do

Apply a function to every reading at once with NumPy's universal functions, write a condition with
`np.where` where a single value would use an `if`, and read the warnings NumPy gives in the places
Python would raise an error.

## The idea

### The problem

Arithmetic on an array already works value by value, and most formulas need more than arithmetic.
The pressure of water vapor in saturated air, which decides when dew and frost form, is an
exponential of the temperature. A standard deviation ends in a square root. Heating degree days, the
measure energy suppliers use to plan for cold weather, take 15.5 minus a day's mean temperature when
the day is colder than 15.5 degrees, and 0 when it is not.

Python's `math` module has `math.exp` and `math.sqrt`, and they take one number. An `if` decides for
one number too. A function written that way runs on Bergen's readings only inside a loop, one reading
at a time, which is the loop arrays were meant to replace.

### What a universal function is

> A **universal function**, or ufunc, is a NumPy function that works value by value on whole arrays.
> It applies one operation to every value, or to every pair of values from two arrays that broadcast
> together, and returns an array of the broadcast shape. `np.sqrt`, `np.exp` and `np.maximum` are
> ufuncs, and so are the arithmetic operators, since `a + b` on arrays runs `np.add(a, b)`.

### Why it works that way

A ufunc runs its loop over the values in compiled code, so `np.exp(readings)` makes 35,040
exponentials in one call. That gives a way of writing formulas, often called elementwise thinking,
in which you write the formula as if for one number, with NumPy's functions in place of `math`'s,
and the same formula then works on an array of any shape. A function that computes the saturation
pressure of one temperature computes it for every reading when it is given the readings.

An `if` is the one piece that does not carry over, because it needs a single `True` or `False`, and a
comparison on an array gives one answer per value. `np.where(condition, a, b)` is the elementwise
form. It takes the value from `a` wherever the condition holds and from `b` everywhere else.

A value with no answer, such as the square root of a negative number or a division by zero, does not
stop the calculation. NumPy puts `nan` or `inf` in that place, warns, and carries on with the rest,
so one bad value among 35,040 does not cost the other answers, and `np.errstate` turns the warnings
into errors for a block of code that should stop at the first bad value.

### Where this shows up

The **Pandas, Deep Dive** guide applies these same functions to the columns of a table. The logistic
function, `1 / (1 + np.exp(-x))`, which turns a score into a probability in the **scikit-learn, Deep
Dive** guide's classification models, is three ufuncs and a division. Image processing keeps pixel
values in range with `np.clip`, and the **SciPy, Deep Dive** guide builds on ufuncs for its special
functions.

### What this notebook covers

- the arithmetic operators, which are ufuncs
- `np.abs`, `np.sqrt` and `np.exp`, and a formula written once for every reading
- `np.round`, `np.floor` and `np.ceil`, and how a value halfway between two others rounds
- `np.clip`, to keep values in a range
- `np.maximum` and `np.minimum` between two arrays
- `np.where` where a single value would use an `if`
- the warnings NumPy gives, and `np.errstate`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read it,
and read the output underneath it. Everything from Setup onward is where you start running things.

```python
import numpy as np

celsius = np.array([-2.0, 4.0, 9.0])
print(np.sqrt(np.abs(celsius)))
print(np.where(celsius < 0, 0.0, celsius))
print(np.maximum(celsius, 3.0))
```

```
[1.41421356 2.         3.        ]
[0. 4. 9.]
[3. 4. 9.]
```

Each function went through all three values without a loop. `np.where` replaced the negative value
with 0, and `np.maximum` raised every value below 3 to 3.

## Setup

Four imports, the year of readings, and a helper that finds an hour's position.

- `numpy`, imported as `np`, is the library, and the cell prints its version
- `math`, `datetime` and `timedelta` build the readings, and `hour_of_year` finds positions
- `warnings` catches the warnings that two of the Common errors give, so that their messages print
  without the name of the file that raised them

`readings_list` holds four lists of 8,760 readings, one for each station in `STATIONS`, made by
`year_of_readings`, the formula the **Why NumPy** notebook explained, and `readings` is the same
values as one array of shape (4, 8760), with `nan` for Svalbard's silent day. `hour_of_year(7, 1, 12)`
gives the position of noon on 1 July along a station's row.

In [1]:
import math
import warnings
from datetime import datetime, timedelta

import numpy as np

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
START = datetime(2025, 1, 1)
HOURS = 365 * 24


def year_of_readings(station):
    """A reading for every hour of 2025 at one station, as the sqlite3 guide makes them.
    Svalbard was silent on 2 March, so those 24 hours are None."""
    i = list(STATIONS).index(station)
    row = []
    for n in range(HOURS):
        hour = START + timedelta(hours=n)
        if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
            row.append(None)
        else:
            season = -math.cos(2 * math.pi * (n - 400) / HOURS)
            day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
            wobble = ((n * 37 + i * 101) % 17 - 8) / 10
            row.append(round(STATIONS[station] + 9 * season + 3 * day + wobble, 1) + 0.0)
    return row


readings_list = [year_of_readings(station) for station in STATIONS]
readings = np.array(readings_list, dtype=float)       # None becomes nan


def hour_of_year(month, day, hour=0):
    """The position along a station's row of an hour of 2025. hour_of_year(7, 1, 12) is noon on 1 July."""
    return (datetime(2025, month, day, hour) - START) // timedelta(hours=1)


print("numpy", np.__version__, "|", len(readings_list), "stations,", len(readings_list[0]), "hours each |", readings.shape)

numpy 2.1.3 | 4 stations, 8760 hours each | (4, 8760)


## Worked examples

### The operators are ufuncs

`+` on an array runs `np.add`. Every ufunc has the type `numpy.ufunc`, and `nin` says how many arrays
it takes:

In [2]:
print(readings[0, :3] + 1.0)
print(np.add(readings[0, :3], 1.0))
print(type(np.add).__name__, type(np.sqrt).__name__)
print(np.add.nin, np.sqrt.nin)

[-2.6 -2.7 -2.7]
[-2.6 -2.7 -2.7]
ufunc ufunc
2 1


The operator and the function gave the same values. `np.add` takes two arrays and `np.sqrt` one.

### np.abs and np.sqrt

Each reading's anomaly is its distance from its own station's mean, from the **Broadcasting**
notebook. `np.abs` drops the sign, and `np.sqrt` takes the square root of every value:

In [3]:
anomaly = readings - np.nanmean(readings, axis=1, keepdims=True)

print(np.nanmean(np.abs(anomaly), axis=1).round(2))
print(np.sqrt(np.nanmean(anomaly ** 2, axis=1)).round(2))
print(np.nanstd(readings, axis=1).round(2))

[5.9 5.9 5.9 5.9]
[6.73 6.73 6.73 6.73]
[6.73 6.73 6.73 6.73]


On average a reading was 5.9 degrees from its station's mean. The square root of the mean squared
anomaly is the standard deviation, 6.73 at every station, which `np.nanstd` computes directly and the
**Aggregation** notebook takes further. The four stations share both numbers because the formula in
Setup gives every station the same swings around a different mean.

### A formula written once with np.exp

One common form of the Magnus formula gives the saturation vapor pressure, in hectopascals, from a
temperature in degrees. Written for one temperature, with `np.exp` in place of `math.exp`, it runs
on any array of temperatures:

In [4]:
def saturation_pressure(celsius):
    return 6.112 * np.exp(17.62 * celsius / (243.12 + celsius))

print(saturation_pressure(np.array([0.0, 10.0, 20.0])).round(2))

pressure = saturation_pressure(readings)
print(pressure.shape, np.isnan(pressure).sum())
print(np.nanmean(pressure, axis=1).round(2))

[ 6.11 12.26 23.33]
(4, 8760) 24
[11.72 10.6   4.9   8.64]


The pressure nearly doubles with every 10 degrees, from 6.11 at 0 to 12.26 at 10 and 23.33 at 20. The
function ran on all 4 by 8,760 readings, and Svalbard's 24 missing hours came out as 24 `nan`
pressures, because every operation in the formula passes `nan` through. Bergen's air could hold the
most water vapor of the four, with a mean of 11.72 hectopascals.

### Rounding with np.round, np.floor and np.ceil

`np.round` takes a number of decimal places. `np.floor` goes down to the whole number below and
`np.ceil` up to the one above:

In [5]:
sample = readings[0, :5]

print(sample)
print(np.round(sample), np.floor(sample), np.ceil(sample))
print(np.round(sample * 9 / 5 + 32, 1))
print(np.round(np.array([0.5, 1.5, 2.5, -3.5])), round(2.5))

[-3.6 -3.7 -3.7 -3.5 -3.1]
[-4. -4. -4. -4. -3.] [-4. -4. -4. -4. -4.] [-3. -3. -3. -3. -3.]
[25.5 25.3 25.3 25.7 26.4]
[ 0.  2.  2. -4.] 2


For negative readings, `np.floor` moves away from zero, so -3.1 became -4. A value exactly halfway
between two whole numbers rounds to the even one, as Python's own `round` does, so 0.5 became 0, 2.5
became 2 and -3.5 became -4. All three functions return decimals, which is why every whole number
prints with a point.

### Keeping values in a range with np.clip

A sensor that reads only from -10 to 15 degrees reports anything colder as -10 and anything warmer as
15. `np.clip` does the same to every reading:

In [6]:
sensor = np.clip(readings, -10, 15)

print(((readings < -10) | (readings > 15)).sum(axis=1))
print(np.nanmin(sensor, axis=1), np.nanmax(sensor, axis=1))
print(np.isnan(sensor).sum(axis=1))

[1621 1043 2358  140]
[ -4.8  -6.3 -10.   -9.2] [15.  15.   8.3 15. ]
[ 0  0 24  0]


The clip changed 2,358 of Svalbard's readings, all of them below -10, and 1,621 of Bergen's, all
above 15. A `nan` stays `nan`, since it is neither below the range nor above it.

### The larger of two arrays with np.maximum

`np.maximum` compares two arrays value by value and keeps the larger, and `np.minimum` keeps the
smaller:

In [7]:
warmer = np.maximum(readings[0], readings[1])

print(readings[0, :3], readings[1, :3])
print(warmer[:3])
print((readings[1] > readings[0]).sum())

[-3.6 -3.7 -3.7] [-3.5 -5.3 -5.3]
[-3.5 -3.7 -3.7]
516


`warmer` holds the warmer of Bergen and Oslo in every hour. Oslo was the warmer of the two for 516
hours of the year.

### np.where where one value would use an if

For a single day, the heating degree days are an `if`:

In [8]:
def heating_needed(day_mean):
    if day_mean < 15.5:
        return 15.5 - day_mean
    return 0.0

print(heating_needed(10.0), heating_needed(18.0))

5.5 0.0


Given an array of days, the `if` would raise the ambiguous truth value error from the **Boolean
Masks** notebook, because the comparison gives one answer per day. `np.where` makes the choice day by
day:

In [9]:
daily = readings.reshape(4, 365, 24).mean(axis=2)
heating = np.where(daily < 15.5, 15.5 - daily, 0.0)

print(daily.shape, heating.shape)
print(heating.sum(axis=1).round(1))
print(np.nansum(np.maximum(15.5 - daily, 0.0), axis=1).round(1))

(4, 365) (4, 365)
[2805.3 3285.1 7273.4 4380. ]
[2805.3 3285.1 7273.4 4380. ]


Bergen needed 2,805.3 heating degree days in 2025, and Svalbard 7,273.4. The larger of a number and 0
is the number when it is positive, so `np.maximum(15.5 - daily, 0.0)` is the same calculation in one
function, and `np.nansum` adds up its days to the same totals.

## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/09-vectorized-math-solutions.ipynb).

**1.** Convert Tromso's readings, row 3, to Fahrenheit, round them to whole degrees with `np.round`, and print the first five.

In [10]:
# your code here

**2.** Print the saturation pressure at -20, -10 and 0 degrees with `saturation_pressure`, rounded to two places.

In [11]:
# your code here

**3.** Use `np.clip` to keep Oslo's readings, row 1, between 0 and 15, and print the smallest and largest value before and after.

In [12]:
# your code here

**4.** Make an array holding the colder of Bergen and Tromso in every hour with `np.minimum`, and print its mean, rounded to two places.

In [13]:
# your code here

**5.** Label each of Oslo's days `"warm"` when its mean is above 10 degrees and `"cool"` otherwise with `np.where`, and count the warm days.

In [14]:
# your code here

## Common errors

### TypeError: only length-1 arrays can be converted to Python scalars

In [15]:
math.sqrt(readings[0])

TypeError: only length-1 arrays can be converted to Python scalars

`math.sqrt` takes one number. Given an array, it tries to turn the array into a single Python number,
and Bergen's 8,760 readings cannot be one number. Use `np.sqrt`, which takes the whole array, and the
same goes for `math.exp`, `math.floor` and the rest of the `math` module.

### TypeError: only integer scalar arrays can be converted to a scalar index

In [16]:
np.max(readings[0], readings[1])

TypeError: only integer scalar arrays can be converted to a scalar index

`np.max` and `np.maximum` are different functions. `np.max` takes one array and reduces it to its
largest value, and its second argument is `axis`, the dimension to reduce along, so NumPy tried to
read Oslo's 8,760 readings as one axis number. `np.maximum(readings[0], readings[1])` compares the two
arrays value by value. The **Aggregation** notebook covers `np.max` and the other functions that
reduce.

### RuntimeWarning: invalid value encountered in sqrt

The standard deviation is the square root of the mean of the squares minus the square of the mean.
Written the other way around, the difference is negative:

In [17]:
bergen = readings[0]

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    spread = np.sqrt(bergen.mean() ** 2 - (bergen ** 2).mean())

print(f"{caught[0].category.__name__}: {caught[0].message}")
print(spread)

nan


A negative number has no real square root, so NumPy gave `nan`, warned, and went on. In a longer
calculation the `nan` would reach the end, and the warning would be the only sign of where it began.
The `warnings` lines catch the warning so that the cell can print its message, and in your own code
it appears under the cell. In the correct order the result is the standard deviation:

In [18]:
print(np.sqrt((bergen ** 2).mean() - bergen.mean() ** 2).round(2))

6.73


### RuntimeWarning: divide by zero encountered in divide

Frost hours for every hour above 10 degrees, a ratio of two counts, has nothing to divide by at
Svalbard, which was never above 10:

In [19]:
frost_hours = (readings < 0).sum(axis=1)
warm_hours = (readings > 10).sum(axis=1)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    ratio = frost_hours / warm_hours

print(f"{caught[0].category.__name__}: {caught[0].message}")
print(frost_hours, warm_hours)
print(ratio.round(2))

[1220 1848 5871 3203] [3718 3197    0 1847]
[0.33 0.58  inf 1.73]


Svalbard's 5,871 frost hours divided by 0 gave `inf`, infinity. `np.errstate` turns the warning into
an error, for a block of code that should stop at the first division by zero:

In [20]:
with np.errstate(divide="raise"):
    frost_hours / warm_hours

FloatingPointError: divide by zero encountered in divide

To leave a station with nothing to divide by as `nan`, give `np.divide` an array to start from with
`out` and a mask of the places to divide with `where`:

In [21]:
ratio = np.divide(frost_hours, warm_hours, out=np.full(4, np.nan), where=warm_hours > 0)

print(ratio.round(2))

[0.33 0.58  nan 1.73]


### No error, and np.where put the missing day in its second branch

In [22]:
labels = np.where(daily >= 0, "thaw", "frost")
march_2 = hour_of_year(3, 2) // 24

print(daily[2, march_2], labels[2, march_2])
print((labels == "frost").sum(axis=1))

nan frost
[ 55  89 243 135]


Svalbard's 2 March has no mean, and `nan >= 0` is `False`, as every comparison with `nan` is, so
`np.where` took the second branch and called a day with no readings a frost day. Svalbard's 243 frost
days include it. Give the missing days a branch of their own:

In [23]:
labels = np.where(np.isnan(daily), "missing", labels)

print((labels == "frost").sum(axis=1))
print((labels == "missing").sum(axis=1))

[ 55  89 242 135]
[0 0 1 0]


## Recap

- Use NumPy's functions, `np.sqrt`, `np.exp` and the rest, in place of the `math` module's on arrays
- Write a formula once for one value, and run it on a whole array
- Round with `np.round`, `np.floor` and `np.ceil`, and keep values in a range with `np.clip`
- Compare two arrays value by value with `np.maximum` and `np.minimum`
- Write a condition with `np.where`, and give `nan` a branch of its own
- Read the warnings for invalid values and division by zero, and raise them with `np.errstate`

## What is next

The **Aggregation** notebook reduces arrays to single values with `sum`, `mean`, `max`, `std` and
`percentile`, and shows what each one does with Svalbard's missing hours.

---

&#8592; **Previous:** [Broadcasting](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/numpy-deep-dive/08-broadcasting.ipynb)  &nbsp;·&nbsp;  [NumPy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/numpy-deep-dive.html)
